In [ ]:
import astropy.units as u
import numpy as np
import naima
from astropy.constants import c
from astropy.io import ascii
import custom_models

In [2]:
filename = 'data/SED2_HESS_100_TeV.txt'

data = ascii.read(filename, format='ipac')

In [3]:
proton_energy = np.logspace(-2,6,1000)*u.GeV

In [ ]:
## Set initial parameters and labels
p0 = np.array((1e38, np.log10(0.5), 3.0))
labels = ["norm", "log10(ref_energy)","index"]

sampler, pos = naima.run_sampler(
    data_table=data,
    p0=p0,
    labels=labels,
    model=custom_models.PionDecay_PL,
    prior=custom_models.lnprior_protons,
    nwalkers=32,
    nburn=300,
    nrun=2000,
    threads=1,
    prefit=False,
)

Burning in the 32 walkers with 300 steps...


Process SpawnPoolWorker-7:
Traceback (most recent call last):
  File "/Users/vdk/miniforge3/envs/naima/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/vdk/miniforge3/envs/naima/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/Users/vdk/miniforge3/envs/naima/lib/python3.12/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/Users/vdk/miniforge3/envs/naima/lib/python3.12/multiprocessing/queues.py", line 389, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'PionDecay_PL' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
Process SpawnPoolWorker-8:
Traceback (most recent call last):
  File "/Users/vdk/miniforge3/envs/naima/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/Users/vdk/miniforge3/envs/naima/lib/py

KeyboardInterrupt: 

In [ ]:
    naima.save_run("SGR_Pion_run.hdf5", sampler)

    naima.save_results_table("SGR_Pion", sampler)

INFO: Saving results table in SGR_Pion_results.ecsv [naima.analysis]


label,median,unc_lo,unc_hi
bytes72,float64,float64,float64
norm,1.2294587771245967e+40,1.1749435622281252e+40,5.905819850847236e+40
log10(ref_energy),-1.1900644689594788,0.35338401331015534,0.5276257156522111
ref_energy,0.06455583918349567,0.03594362168404015,0.15299524235347878
index,2.7247140743076907,0.20092704436125608,0.22308194452864782
blob2,2.0162894778904595e+51,1.6016538376490704e+51,9.647022875749795e+51
blob4,4.0325789557809143e+49,3.2033076752981376e+49,1.9294045751499583e+50


In [ ]:
    naima.save_diagnostic_plots(
        "SGR_Pion",
        sampler,
        sed=True,
        last_step=False,
        blob_labels=[
            "Spectrum",
            "Electron energy distribution",
            "$W_e (E_e>1\, \mathrm{KeV})$",
        ],
    )

In [ ]:
# Приклад витягу медіан і похибок (після фіту)
    median_params = np.median(sampler.flatchain, axis=0)
    errors = np.percentile(sampler.flatchain, [16, 84], axis=0)
    print("Median parameters:", dict(zip(labels, median_params)))
    print("Errors (1-sigma):", errors)

In [ ]:
# Wp і We з блобів (медіани)
    Wp_blob = sampler.blobs[..., 2].flatten()  # Третій блоб — Wp
    We_blob = sampler.blobs[..., 4].flatten()  # П'ятий блоб — We
    median_Wp = np.median(Wp_blob)
    median_We = np.median(We_blob)
    print(f"Median Wp: {median_Wp}, We: {median_We} (We/Wp ≈ {median_We / median_Wp})")